# Exercise 6: Generative Adversarial Networks
## Teaching Two Networks to Outsmart Each Other

---

### Learning Objectives
- Understand the adversarial game: Generator vs Discriminator
- Implement a vanilla GAN and watch it fail (mode collapse)
- Train a **DCGAN** to generate realistic handwritten digits
- Explore the latent space through interpolation
- Understand common GAN failure modes and how to diagnose them

### The Core Idea
Two neural networks locked in a game:
- **Generator G**: Takes random noise $z \sim p_z$ → produces fake images
- **Discriminator D**: Takes an image → outputs P(real)

$$\min_G \max_D \; \mathbb{E}_{x \sim p_{data}} [\log D(x)] + \mathbb{E}_{z \sim p_z} [\log(1 - D(G(z)))]$$

At equilibrium, G generates images indistinguishable from real data.

---

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
import os
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

LATENT_DIM = 100
IMG_SIZE = 28
CHANNELS = 1
plt.style.use('dark_background')

In [ ]:
# Load MNIST
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

dataset = torchvision.datasets.MNIST('./data', train=True, download=True, transform=transform)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=128, shuffle=True, num_workers=2)

# Visualize real samples
real_batch = next(iter(dataloader))
plt.figure(figsize=(12, 3))
plt.title('Real MNIST Samples', fontsize=14, fontweight='bold', color='white')
img_grid = torchvision.utils.make_grid(real_batch[0][:16], nrow=8, normalize=True)
plt.imshow(img_grid.permute(1, 2, 0).numpy(), cmap='gray')
plt.axis('off')
plt.tight_layout()
plt.show()
print(f'Dataset size: {len(dataset):,} images')

## Part 1: Vanilla GAN — Watch It Struggle

First, let's build the simplest possible GAN with only fully-connected layers. This will suffer from **mode collapse** and **training instability** — we'll fix it in Part 2 with DCGAN.

In [ ]:
class VanillaGenerator(nn.Module):
    def __init__(self, latent_dim=100, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 256), nn.LeakyReLU(0.2),
            nn.Linear(256, 512),        nn.LeakyReLU(0.2),
            nn.Linear(512, 1024),       nn.LeakyReLU(0.2),
            nn.Linear(1024, img_dim),   nn.Tanh()
        )

    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)


class VanillaDiscriminator(nn.Module):
    def __init__(self, img_dim=784):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(img_dim, 1024), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(1024, 512),     nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256),      nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1),        nn.Sigmoid()
        )

    def forward(self, img):
        return self.net(img.view(img.size(0), -1))


vanilla_G = VanillaGenerator(LATENT_DIM).to(device)
vanilla_D = VanillaDiscriminator().to(device)
print(f'Generator params:     {sum(p.numel() for p in vanilla_G.parameters()):,}')
print(f'Discriminator params: {sum(p.numel() for p in vanilla_D.parameters()):,}')

In [ ]:
def train_gan(G, D, dataloader, epochs=30, lr=2e-4, name='GAN'):
    G_opt = optim.Adam(G.parameters(), lr=lr, betas=(0.5, 0.999))
    D_opt = optim.Adam(D.parameters(), lr=lr, betas=(0.5, 0.999))
    criterion = nn.BCELoss()
    
    fixed_noise = torch.randn(64, LATENT_DIM, device=device)
    G_losses, D_losses = [], []
    image_snapshots = []
    
    for epoch in range(epochs):
        for real_imgs, _ in dataloader:
            batch_size = real_imgs.size(0)
            real_imgs = real_imgs.to(device)
            real_labels = torch.ones(batch_size, 1, device=device)
            fake_labels = torch.zeros(batch_size, 1, device=device)
            
            # === Train Discriminator ===
            D.zero_grad()
            z = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = G(z).detach()
            
            d_real = D(real_imgs)
            d_fake = D(fake_imgs)
            d_loss = criterion(d_real, real_labels) + criterion(d_fake, fake_labels)
            d_loss.backward()
            D_opt.step()
            
            # === Train Generator ===
            G.zero_grad()
            z = torch.randn(batch_size, LATENT_DIM, device=device)
            fake_imgs = G(z)
            g_loss = criterion(D(fake_imgs), real_labels)
            g_loss.backward()
            G_opt.step()
            
            G_losses.append(g_loss.item())
            D_losses.append(d_loss.item())
        
        with torch.no_grad():
            fake = G(fixed_noise).cpu()
        image_snapshots.append(fake)
        
        if (epoch + 1) % 5 == 0:
            print(f'[{name}] Epoch {epoch+1:3d}/{epochs} | G_loss: {np.mean(G_losses[-100:]):.4f} | D_loss: {np.mean(D_losses[-100:]):.4f}')
    
    return G_losses, D_losses, image_snapshots, fixed_noise

print('Training Vanilla GAN (watch for instability and mode collapse)...')
v_G_losses, v_D_losses, v_snapshots, fixed_z = train_gan(vanilla_G, vanilla_D, dataloader, epochs=20, name='VanillaGAN')

## Part 2: DCGAN — Deep Convolutional GAN

Radford et al. (2015) showed that replacing fully-connected layers with **strided convolutions** dramatically stabilizes training.

Key DCGAN principles:
1. Generator uses **transposed convolutions** (upsampling)
2. Discriminator uses **strided convolutions** (downsampling) instead of pooling
3. **Batch Normalization** in both networks (except D's input and G's output)
4. **LeakyReLU** in D, **ReLU** in G

In [ ]:
def weights_init(m):
    """Custom weight initialization from DCGAN paper."""
    classname = m.__class__.__name__
    if 'Conv' in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)


class DCGenerator(nn.Module):
    """
    Takes z (100,) → 28×28 image via transposed convolutions.
    Architecture: z → Linear → Reshape → [TransConv → BN → ReLU] × 3 → Tanh
    """
    def __init__(self, latent_dim=100, ngf=64):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(latent_dim, ngf * 4 * 7 * 7),
            nn.BatchNorm1d(ngf * 4 * 7 * 7),
            nn.ReLU()
        )
        self.conv_blocks = nn.Sequential(
            # 7×7 → 14×14
            nn.ConvTranspose2d(ngf*4, ngf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf*2),
            nn.ReLU(True),
            # 14×14 → 28×28
            nn.ConvTranspose2d(ngf*2, ngf, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ngf),
            nn.ReLU(True),
            # Polish
            nn.Conv2d(ngf, 1, 3, 1, 1, bias=False),
            nn.Tanh()
        )
        self.ngf = ngf

    def forward(self, z):
        x = self.project(z).view(-1, self.ngf * 4, 7, 7)
        return self.conv_blocks(x)


class DCDiscriminator(nn.Module):
    """
    Takes 28×28 image → scalar probability via strided convolutions.
    Architecture: [Conv → BN → LeakyReLU] × 3 → Linear → Sigmoid
    """
    def __init__(self, ndf=64):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            # 28×28 → 14×14
            nn.Conv2d(1, ndf, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # 14×14 → 7×7
            nn.Conv2d(ndf, ndf*2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*2),
            nn.LeakyReLU(0.2, inplace=True),
            # 7×7 → 3×3 
            nn.Conv2d(ndf*2, ndf*4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(ndf*4),
            nn.LeakyReLU(0.2, inplace=True),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(ndf*4 * 3 * 3, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.classifier(self.conv_blocks(x))


dcG = DCGenerator(LATENT_DIM).to(device)
dcD = DCDiscriminator().to(device)
dcG.apply(weights_init)
dcD.apply(weights_init)

print(f'DCGAN Generator params:     {sum(p.numel() for p in dcG.parameters()):,}')
print(f'DCGAN Discriminator params: {sum(p.numel() for p in dcD.parameters()):,}')

# Verify output shapes
test_z = torch.randn(4, LATENT_DIM).to(device)
test_img = dcG(test_z)
print(f'Generator output shape: {test_img.shape}')  # Should be [4, 1, 28, 28]
print(f'Discriminator output:   {dcD(test_img).shape}')  # Should be [4, 1]

In [ ]:
print('Training DCGAN...')
dc_G_losses, dc_D_losses, dc_snapshots, fixed_z_dc = train_gan(dcG, dcD, dataloader, epochs=30, lr=2e-4, name='DCGAN')

In [ ]:
# Compare Vanilla GAN vs DCGAN
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss curves
for ax, (g_losses, d_losses, title) in zip(axes[:, 0], [
    (v_G_losses, v_D_losses, 'Vanilla GAN Losses'),
    (dc_G_losses, dc_D_losses, 'DCGAN Losses'),
]):
    window = 200
    smooth_g = np.convolve(g_losses, np.ones(window)/window, mode='valid')
    smooth_d = np.convolve(d_losses, np.ones(window)/window, mode='valid')
    ax.plot(smooth_g, label='Generator', color='#e74c3c', linewidth=2)
    ax.plot(smooth_d, label='Discriminator', color='#2ecc71', linewidth=2)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')

# Final generated samples
for ax, (snapshots, model, name) in zip(axes[:, 1:].reshape(2, -1)[:, :2], [
    (v_snapshots, vanilla_G, 'Vanilla GAN'),
    (dc_snapshots, dcG, 'DCGAN'),
]):
    model.eval()
    with torch.no_grad():
        if name == 'Vanilla GAN':
            imgs = vanilla_G(fixed_z).cpu()
        else:
            imgs = dcG(fixed_z_dc).cpu()
    grid = torchvision.utils.make_grid(imgs[:16], nrow=4, normalize=True)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(f'{name}\nFinal Generated Images', fontsize=11, fontweight='bold')
    ax.axis('off')

# Early vs Late snapshots for DCGAN
for ax, (snap_idx, label) in zip(axes[1, 1:], [
    (0, 'DCGAN at Epoch 1'),
    (len(dc_snapshots)-1, 'DCGAN at Epoch 30'),
]):
    grid = torchvision.utils.make_grid(dc_snapshots[snap_idx][:16], nrow=4, normalize=True)
    ax.imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.axis('off')

plt.suptitle('Vanilla GAN vs DCGAN Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Part 3: Latent Space Interpolation — The Geometry of Images

One of the most beautiful properties of GANs: the latent space is **smooth and continuous**. You can interpolate between two noise vectors and watch one image gradually morph into another.

In [ ]:
def slerp(val, low, high):
    """Spherical linear interpolation — better than linear for high-dim spherical latent spaces."""
    low_norm = low / torch.norm(low, dim=1, keepdim=True)
    high_norm = high / torch.norm(high, dim=1, keepdim=True)
    omega = torch.acos(torch.clamp((low_norm * high_norm).sum(1), -1, 1))
    so = torch.sin(omega)
    # Fallback to lerp if omega is near 0
    cond = (so < 1e-10).unsqueeze(1)
    res = (torch.sin((1.0-val)*omega) / so).unsqueeze(1) * low + \
          (torch.sin(val*omega) / so).unsqueeze(1) * high
    return torch.where(cond, (1-val) * low + val * high, res)


dcG.eval()
n_steps = 10
n_pairs = 4

torch.manual_seed(2024)
z1 = torch.randn(n_pairs, LATENT_DIM, device=device)
z2 = torch.randn(n_pairs, LATENT_DIM, device=device)

fig, axes = plt.subplots(n_pairs, n_steps, figsize=(22, 8))

with torch.no_grad():
    for pair in range(n_pairs):
        z_a = z1[pair:pair+1]
        z_b = z2[pair:pair+1]
        for step in range(n_steps):
            t = step / (n_steps - 1)
            z_interp = (1 - t) * z_a + t * z_b  # Linear interpolation
            img = dcG(z_interp).squeeze().cpu().numpy()
            axes[pair][step].imshow(img, cmap='gray', vmin=-1, vmax=1)
            axes[pair][step].axis('off')
            if pair == 0:
                axes[pair][step].set_title(f't={t:.1f}', fontsize=9, color='white')

axes[0][0].set_ylabel('←  Start', color='white', fontsize=11)
axes[0][-1].set_ylabel('End  →', color='white', fontsize=11)

plt.suptitle('Latent Space Interpolation\n(Each row: smooth morphing between two random digits)', 
             fontsize=14, fontweight='bold', color='white')
plt.tight_layout()
plt.show()

## Part 4: Diagnosing Mode Collapse

**Mode collapse**: The generator finds a few "safe" outputs that fool the discriminator and only generates those. A GAN should generate ALL 10 digit classes — if it collapses, it might only generate 2-3.

In [ ]:
def diagnose_mode_collapse(G, n_samples=1000, name='GAN'):
    """Check if the generator produces diverse outputs."""
    G.eval()
    
    # Generate many samples
    all_images = []
    batch = 256
    for _ in range(n_samples // batch):
        z = torch.randn(batch, LATENT_DIM, device=device)
        with torch.no_grad():
            imgs = G(z).cpu()
        all_images.append(imgs)
    all_images = torch.cat(all_images)
    
    # Load a pretrained MNIST classifier to check class distribution
    # (Quick proxy: just check visual diversity via pixel variance)
    pixel_means = all_images.view(len(all_images), -1).mean(dim=1)
    pixel_stds  = all_images.view(len(all_images), -1).std(dim=1)
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    
    # Distribution of mean brightness
    axes[0].hist(pixel_means.numpy(), bins=50, color='#3498db', edgecolor='white', alpha=0.8)
    axes[0].set_title('Pixel Mean Distribution\n(narrow = mode collapse)', fontweight='bold')
    axes[0].set_xlabel('Mean pixel value')
    
    # Distribution of std dev
    axes[1].hist(pixel_stds.numpy(), bins=50, color='#e74c3c', edgecolor='white', alpha=0.8)
    axes[1].set_title('Pixel Std Distribution\n(narrow = mode collapse)', fontweight='bold')
    axes[1].set_xlabel('Std pixel value')
    
    # Grid of 32 generated samples
    grid = torchvision.utils.make_grid(all_images[:32], nrow=8, normalize=True)
    axes[2].imshow(grid.permute(1, 2, 0).numpy(), cmap='gray')
    axes[2].set_title('Random Sample Grid\n(variety of digits?)', fontweight='bold')
    axes[2].axis('off')
    
    diversity = pixel_means.std().item()
    plt.suptitle(f'{name} — Diversity Score (σ of means): {diversity:.4f}\n'
                 f'(Higher = more diverse, lower = mode collapse)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    return diversity

print('=== Vanilla GAN Diversity ===')
v_diversity = diagnose_mode_collapse(vanilla_G, name='Vanilla GAN')

print('\n=== DCGAN Diversity ===')
dc_diversity = diagnose_mode_collapse(dcG, name='DCGAN')

print(f'\nVanilla GAN diversity: {v_diversity:.4f}')
print(f'DCGAN diversity:       {dc_diversity:.4f}')
print(f'DCGAN is {dc_diversity/v_diversity:.1f}× more diverse' if v_diversity > 0 else '')

## Exercises

### Exercise A — Conditional GAN (cGAN)
A regular GAN generates random digits. A **conditional GAN** lets you control which digit to generate by conditioning on a class label:
$$G(z, y) \rightarrow \text{image of digit } y$$

Implement a cGAN by:
1. Adding label embeddings to both G and D
2. G receives `[z, embed(y)]` as input
3. D receives `[image, embed(y)]` concatenated
4. Test: generate a grid of each digit 0-9

### Exercise B — Wasserstein GAN (Conceptual)
The original GAN loss can lead to vanishing gradients when D is too good. Wasserstein GAN (WGAN) uses a different loss:
$$\min_G \max_{D \in \mathcal{1-Lip}} \mathbb{E}[D(x)] - \mathbb{E}[D(G(z))]$$

Implement WGAN-GP (Gradient Penalty variant):
1. Remove sigmoid from D
2. Replace BCE loss with: `d_loss = D(fake) - D(real) + λ * gradient_penalty`
3. Add gradient penalty: `||∇D(x̂)||₂ - 1)²` on interpolated samples
4. Compare training stability vs vanilla GAN

### Exercise C — FID Score
**Fréchet Inception Distance (FID)** measures the quality of generated images. Implement it:
1. Extract features from real and generated images using a pretrained InceptionV3
2. Compute mean and covariance of both feature distributions
3. FID = $||\mu_r - \mu_g||^2 + \text{Tr}(\Sigma_r + \Sigma_g - 2(\Sigma_r\Sigma_g)^{1/2})$
4. Lower FID = better. SOTA on MNIST is ~1-3.

### Discussion Questions
1. The GAN training objective is a **minimax game**. What is the Nash equilibrium? Is it stable?
2. Why does DCGAN use Batch Normalization but NOT in the Generator's output layer or the Discriminator's input layer?
3. Look at the latent space interpolation. The images change smoothly — what does this tell us about the geometry of the learned representation?